# 📓 Lab 01 — Tensores, shapes, broadcasting y autograd

**Curso:** Deep Learning: de los fundamentos a los Transformers · **Sesión 1**
**Material del aula:** [Sesión 1 — Fundamentos](../sesiones/01-fundamentos.md)

En este notebook vas a:
1. Crear tensores y leer sus shapes en voz alta.
2. Ver la vectorización y el broadcasting en acción.
3. Verificar a mano las derivadas que calcula `autograd`.

> **Regla del curso:** predice el resultado ANTES de ejecutar cada celda.

In [ ]:
# Verificación del entorno: si esto corre, estamos listos
import sys
import torch

print('Python :', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
print('MPS    :', hasattr(torch.backends, 'mps') and torch.backends.mps.is_available())

## 1. Tensores y shapes

Un tensor es una caja de números con ejes. El **shape es un contrato**: describe qué
significa cada eje. Aprende a leerlos en voz alta:

| Shape | Lectura |
|---|---|
| `()` | un escalar |
| `(3,)` | un vector de 3 |
| `(32, 10)` | batch de 32 muestras, 10 features |
| `(32, 3, 224, 224)` | batch de 32 imágenes RGB de 224×224 |

In [ ]:
# Escalar, vector, matriz, tensor 4D — y sus shapes
escalar = torch.tensor(3.14)
vector = torch.tensor([1.0, 2.0, 3.0])
matriz = torch.randn(32, 10)          # batch de 32 muestras, 10 features
imagenes = torch.randn(32, 3, 224, 224)  # batch de imágenes NCHW = (batch, canales, alto, ancho)

for nombre, t in [('escalar', escalar), ('vector', vector),
                  ('matriz', matriz), ('imagenes', imagenes)]:
    print(f'{nombre:9s} shape={tuple(t.shape)}  ndim={t.ndim}  dtype={t.dtype}')

### 🧩 Predice antes de ejecutar

¿Qué shape tiene `matriz.mean(dim=0)`? ¿Y `matriz.mean(dim=1)`?
`dim=k` significa "colapsa el eje k". Escribe tu respuesta y luego ejecuta.

In [ ]:
# ¿Acertaste?
print('mean(dim=0):', matriz.mean(dim=0).shape)  # promedio POR FEATURE  → (10,)
print('mean(dim=1):', matriz.mean(dim=1).shape)  # promedio POR MUESTRA  → (32,)

## 2. Vectorización y broadcasting

*Broadcasting*: PyTorch "estira" automáticamente el tensor pequeño para que las shapes coincidan (aquí, el escalar `b` se suma a cada elemento).

La demo central de la Sesión 1: una capa lineal para un batch completo,
sin ningún `for`.

In [ ]:
# Batch de 2 muestras con 3 features cada una
x = torch.tensor([
    [1.0, 2.0, 3.0],
    [4.0, 5.0, 6.0],
])                                    # (2, 3)

w = torch.tensor([0.2, -0.1, 0.5])    # (3,) un peso por feature
b = torch.tensor(0.3)                 # escalar

# logit = salida cruda del modelo, antes de convertir a probabilidad (Sesión 1 §3)
logits = x @ w + b                    # @ = producto matricial; b se broadcast
print('x     :', x.shape)
print('w     :', w.shape)
print('logits:', logits.shape)        # (2,) → un logit por muestra
print(logits)

**Preguntas** (respóndelas en esta celda):

1. ¿Qué representa cada eje de `x`?
2. ¿Por qué `x @ w` produce shape `(2,)`?
3. ¿Qué cambiaría si `w` fuera `(3, 2)`?

*Tus respuestas aquí...*

In [ ]:
# Verifica tu respuesta a la pregunta 3:
W = torch.randn(3, 2)                 # ahora hay DOS neuronas (2 columnas)
salida = x @ W + torch.randn(2)       # (2,3) @ (3,2) → (2,2)
print('salida:', salida.shape)        # dos logits por muestra

## 3. Vectorizar vs bucle: por qué importa

Misma operación, dos implementaciones. La diferencia de tiempo es la razón
por la que TODO en Deep Learning se hace por lotes.

In [ ]:
import time

X = torch.randn(2000, 512)
W2 = torch.randn(512, 512)

# Versión con bucle (procesar muestra por muestra)
inicio = time.perf_counter()
resultado_bucle = torch.stack([X[i] @ W2 for i in range(len(X))])
t_bucle = time.perf_counter() - inicio

# Versión vectorizada (el batch completo de una vez)
inicio = time.perf_counter()
resultado_vec = X @ W2
t_vec = time.perf_counter() - inicio

assert torch.allclose(resultado_bucle, resultado_vec, atol=1e-4)
print(f'bucle       : {t_bucle*1000:7.1f} ms')
print(f'vectorizado : {t_vec*1000:7.1f} ms')
print(f'aceleración : {t_bucle/t_vec:.0f}×')

## 4. Autograd visible

El experimento clave de la sesión: PyTorch calcula gradientes automáticamente
construyendo un **grafo computacional** durante el forward.

Modelo mínimo: $\hat{y} = wx + b$, pérdida $L = (\hat{y} - y)^2$.

Derivadas manuales (regla de la cadena):

$$\frac{\partial L}{\partial w} = 2(\hat{y} - y)\,x
\qquad \frac{\partial L}{\partial b} = 2(\hat{y} - y)$$

In [ ]:
# Los tensores con requires_grad=True son "rastreados" por autograd
x = torch.tensor(2.0)
w = torch.tensor(3.0, requires_grad=True)   # parámetro
b = torch.tensor(1.0, requires_grad=True)   # parámetro
y_true = torch.tensor(10.0)

y_pred = w * x + b              # forward: construye el grafo
loss = (y_pred - y_true) ** 2   # L = (7 - 10)² = 9

loss.backward()                 # backward: propaga gradientes hacia atrás

print('y_pred :', y_pred.item())      # 7.0
print('loss   :', loss.item())        # 9.0
print('dL/dw  :', w.grad.item())      # 2·(7−10)·2 = −12  ✓
print('dL/db  :', b.grad.item())      # 2·(7−10)   = −6   ✓

### ✅ Verificación manual

Comprueba que autograd coincide con la derivación:
- $\partial L/\partial w = 2(7-10)\cdot 2 = -12$ ✓
- $\partial L/\partial b = 2(7-10) = -6$ ✓

### ⚠️ Los gradientes se ACUMULAN

Ejecuta la siguiente celda dos veces y observa qué pasa con `w.grad`.
Esta es la razón del `optimizer.zero_grad()` en todo training loop.

In [ ]:
# Ejecutar esta celda DOS veces: el gradiente se suma, no se reemplaza
loss = (w * x + b - y_true) ** 2
loss.backward()
print('w.grad acumulado:', w.grad.item())

# Limpieza manual (lo que hace optimizer.zero_grad()):
w.grad = None
b.grad = None

## 5. Un paso de gradient descent, a mano

Cerramos el círculo: usar el gradiente para MEJORAR los parámetros.

In [ ]:
# Descenso por gradiente manual: θ ← θ − η·∇L
w = torch.tensor(3.0, requires_grad=True)
b = torch.tensor(1.0, requires_grad=True)
lr = 0.05   # learning rate

for paso in range(15):
    loss = (w * x + b - y_true) ** 2       # forward
    loss.backward()                        # gradientes

    with torch.no_grad():                  # actualizar SIN rastrear el update
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad = None                          # limpiar para el siguiente paso
    b.grad = None

    if paso % 3 == 0:
        print(f'paso {paso:2d} | loss={loss.item():8.4f} | '
              f'w={w.item():.3f} b={b.item():.3f}')

print(f'\nPredicción final: {(w * x + b).item():.3f}  (objetivo: 10.0)')

## 🎯 Reto final

1. Cambia el learning rate a `1.0`. ¿Qué pasa? ¿Por qué?
2. Cámbialo a `0.001`. ¿Cuántos pasos necesitarías ahora?
3. Relaciona lo observado con la figura de learning rates de la
   [Sesión 1](../sesiones/01-fundamentos.md) y con el
   [simulador de descenso de gradiente](https://felmco.github.io/deeplearning-class/interactivos/descenso-gradiente.html).

**Commit sugerido:** `feat: complete tensors and autograd lab`